In [1]:
%pip install -q -e .[train]

Note: you may need to restart the kernel to use updated packages.


C:\Users\Alexander\PycharmProjects\ECup\.venv\Scripts\python.exe: No module named pip


In [2]:
import mlflow
import torch
import polars as pl
from dotenv import load_dotenv
from transformers import AutoTokenizer
from tqdm.autonotebook import tqdm
from sklearn.metrics import average_precision_score
from peft import LoraConfig, get_peft_model

from utility.model import HFCrossEncoder, product_text
from utility.sampling import train_test_split
from utility.eval import macro_pr_auc
from utility import load

C:\Users\Alexander\PycharmProjects\ECup\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Alexander\PycharmProjects\ECup\.venv\Lib\site-packages\torchao\quantization\quant_api.py:1558: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
W0828 16:30:35.818000 18080 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [3]:
load_dotenv()
mlflow.set_experiment('ecup-student-fine-tuning')

exp = mlflow.get_experiment_by_name('ecup-fine-tuning')
env = load()
repo_url = 'hf://datasets/' + env.config.data.data_repo
model_config = env.config.model.params
print(model_config)
print('experiment:', exp.experiment_id)

{'student_name': 'well-please/student_model', 'teacher_name': 'well-please/teacher_model', 'tokenizer_name': 'well-please/teacher_model', 'seed': 69, 'teacher_epochs': 2, 'student_epochs': 2, 'teacher_batch': 128, 'student_batch': 128, 'inference_batch': 512, 'lr': 1e-05, 'max_len': 320, 'alpha': 0.5, 'test_size': 0.2}
experiment: 2


In [4]:
ATTR_CAP = 1500
TOTAL_CAP = 2000
SEED = model_config['seed']
TEST_SIZE = model_config['test_size']

TEACHER_EPOCHS = model_config['teacher_epochs']
STUDENT_EPOCHS = model_config['student_epochs']
TEACHER_BATCH = model_config['teacher_batch']
STUDENT_BATCH = model_config['student_batch']
INFERENCE_BATCH = model_config['inference_batch']
ANSWER_BATCH = 256
LR = model_config['lr']
MAX_LEN = model_config['max_len']

ALPHA = model_config['alpha']
ANSWER_COL = 'teacher_answer'

DEVICE = 'cuda' if torch.cuda.is_available() else ('xpu' if torch.xpu.is_available() else 'cpu')

In [5]:
student = HFCrossEncoder.from_pretrained(model_config['student_name']).to(DEVICE, dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_config['tokenizer_name'])

C:\Users\Alexander\PycharmProjects\ECup\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Alexander\.cache\huggingface\hub\models--well-please--student_model. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10186.75it/s]
C:\Users\Alexander\Pych

In [6]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['clf', 'query', 'key', 'value'],
    layers_to_transform=list(range(6, 12)),
    lora_dropout=0.05,
)

student = get_peft_model(student, lora_config)

In [7]:
items = pl \
    .scan_parquet(f'{repo_url}/{env.config.data.items}') \
    .select(
        'id',
        pl.struct(['name', 'category', 'attributes'])
            .map_elements(lambda r: product_text(r['name'], r['category'], r['attributes'], attr_cap=ATTR_CAP, total_cap=TOTAL_CAP), return_dtype=pl.String).alias('text'),
        'category',
    )
items_human = pl \
    .scan_parquet(f'{repo_url}/{env.config.data.items_human}') \
    .select(
        'id',
        pl.struct(['name', 'category', 'attributes'])
            .map_elements(lambda r: product_text(r['name'], r['category'], r['attributes'], attr_cap=ATTR_CAP, total_cap=TOTAL_CAP), return_dtype=pl.String).alias('text'),
        'category'
    )
matches = pl.scan_parquet(f'{repo_url}/{env.config.data.matches}')
matches_llm = pl.scan_parquet(f'{repo_url}/{env.config.data.matches_llm}')

human_pairs = matches \
    .join(items_human.select('id', pl.col('text').alias('text1'), pl.col('category').alias('category1')), right_on='id', left_on='id1') \
    .join(items_human.select('id', pl.col('text').alias('text2'), pl.col('category').alias('category2')), right_on='id', left_on='id2') \
    .unique()

llm_pairs = matches_llm \
    .join(items.select('id', pl.col('text').alias('text1'), pl.col('category').alias('category1')), right_on='id', left_on='id1') \
    .join(items.select('id', pl.col('text').alias('text2'), pl.col('category').alias('category2')), right_on='id', left_on='id2') \
    .unique()

In [8]:
stratification_columns = ('category1', 'category2', 'target')
train, test = train_test_split(human_pairs, stratification_columns, TEST_SIZE, SEED)
train, test = train.collect() if isinstance(train, pl.LazyFrame) else train, test.collect() if isinstance(test, pl.LazyFrame) else test
train = train.sort(pl.col('text1').str.len_chars() + pl.col('text2').str.len_chars())

dataset_metadata = {
    'seed': SEED,
    'test_size': TEST_SIZE,
    'stratification_columns': stratification_columns,
    'train_shape': train.shape,
    'test_shape': test.shape
}

train.height, test.height

(229645, 14910)

In [9]:
def apply_tokenizer(data_slice):
    return tokenizer(
        data_slice['text1'].to_list(),
        data_slice['text2'].to_list(),
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN,
        return_tensors='pt',
    )

In [10]:
def train_epoch(model, scaler, optimizer, criterion, training_data, batch_size, current_epoch):
    epoch_acc_loss = 0.
    epoch_steps = 0
    steps_per_epoch = (training_data.height + batch_size - 1) // batch_size
    for batch_idx in tqdm(range(0, training_data.height, batch_size), desc='training epoch'):
        text_batch = training_data.slice(batch_idx, batch_size).select('target', 'text1', 'text2', 'category1')
        batch = apply_tokenizer(text_batch).to(DEVICE)
        target = text_batch['target'].to_torch().float().to(DEVICE)

        with torch.autocast(device_type=DEVICE, dtype=torch.float16):
            logits = model(**batch).float()

        loss = criterion(logits, target)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(optimizer.param_groups[0]['params'], max_norm=10.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        mlflow.log_metric('epoch_loss', loss.item(), step=(current_epoch - 1) * steps_per_epoch + batch_idx // batch_size)

        epoch_acc_loss += loss.item()
        epoch_steps += 1

        if epoch_steps % 25 == 0:
            metric = macro_pr_auc(target, logits, text_batch['category1'].to_numpy())
            mlflow.log_metric('mean_pc-auc', metric, step=(current_epoch - 1) * (steps_per_epoch // 25) + batch_idx // batch_size // 25)

    print(f'epoch loss: {epoch_acc_loss / epoch_steps:.2f} on epoch {current_epoch}')
    return epoch_acc_loss / epoch_steps

In [11]:
def student_distillation(training_data: pl.DataFrame):
    global student

    criterion = torch.nn.BCEWithLogitsLoss().to(DEVICE)
    scaler = torch.amp.GradScaler(device=DEVICE)
    optim = torch.optim.AdamW(student.get_active_params(), lr=LR)

    with mlflow.start_run(nested=True, run_name='student distillation', tags={'mlflow.user': 'jstnoname'}):
        for epoch in tqdm(range(1, STUDENT_EPOCHS + 1), desc='student epochs'):
            epoch_loss = train_epoch(student, scaler, optim, criterion, training_data, STUDENT_BATCH, epoch)
            mlflow.log_metric('total_epoch_loss', epoch_loss, step=epoch)

    student = student.merge_and_unload()
    student.push_to_hub('well-please/student_model', private=True, commit_message='student distillation')
    mlflow.transformers.log_model(student, torch_dtype=torch.float16)

In [12]:
@torch.inference_mode()
def validation(test_data: pl.DataFrame):
    student.eval()

    teacher_scores = []
    student_scores = []

    with mlflow.start_run(nested=True, run_name='validation', tags={'mlflow.user': 'jstnoname'}):
        for batch_idx in tqdm(range(0, test_data.height, INFERENCE_BATCH), desc='validation'):
            text_batch = test_data.slice(batch_idx, INFERENCE_BATCH).select('target', 'text1', 'text2')
            batch = apply_tokenizer(text_batch).to(DEVICE)

            student_scores.append(student(**batch).cpu())

        teacher_scores = torch.cat(teacher_scores).float().numpy()
        student_scores = torch.cat(student_scores).float().numpy()
        y_true = test_data['target'].to_numpy()
        categories = test_data['category1'].to_numpy()

        metrics = {
            'test.teacher.global_ap': average_precision_score(y_true, teacher_scores),
            'test.teacher.macro_pr_auc': macro_pr_auc(y_true, teacher_scores, categories),
            'test.student.global_ap': average_precision_score(y_true, student_scores),
            'test.student.macro_pr_auc': macro_pr_auc(y_true, student_scores, categories),
        }
        mlflow.log_metrics(metrics)

    print('\n'.join(list(map(lambda pair: f'{pair[0]}: {pair[1]}', metrics.items()))))
    return metrics

In [ ]:
with mlflow.start_run(run_name='student fine-tuning', tags={'mlflow.user': 'jstnoname'}):
    mlflow.log_params({
        'seed': SEED,
        'lr': LR,
        'student_epochs': STUDENT_EPOCHS,
        'student_batch': STUDENT_BATCH,
        'alpha': ALPHA,
        'dataset_metadata': ' | '.join(list(map(lambda pair: f'{pair[0]}: {pair[1]}', dataset_metadata.items())))
    })

    student_distillation(train)
    validation(test)

training epoch:   0%|          | 0/1795 [00:00<?, ?it/s]